# 드럼 분리 벤치마크 — 코랩 실행

저장소 `AI/` 코드와 구글 드라이브의 데이터로 `ssaplash-bench` 명령을 그대로 돌린다 (설계 4-3절, ADR-008·011).
셀마다 명령만 부르고 로직은 없다. 결과는 드라이브에 저장한다.

준비물 (드라이브 `MyDrive/ssaplash/`)
- `S15P31A307.zip`: 저장소를 zip으로 묶은 것 (미결 U7: 1차는 zip 업로드)
- `data/musdb18hq/{train,test}/...`, `data/gmd/groove/...`, `data/synth_v1/...`: 로컬에서 `AI/data/` 그대로 올린 것

런타임은 GPU(코랩 프로+ A100 또는 L4)로 바꾼 뒤 위에서부터 실행한다.


In [ ]:
# 1. 드라이브 연결
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ssaplash"
!ls -la $DRIVE


In [ ]:
# 2. 저장소 풀기 (zip 방식). 이미 풀려 있으면 건너뜀
import os
if not os.path.isdir("/content/S15P31A307/AI"):
    !unzip -q "$DRIVE/S15P31A307.zip" -d /content/
%cd /content/S15P31A307/AI
!git log --oneline -1 2>/dev/null || echo "git 정보 없음 (zip)"


In [ ]:
# 3. uv 설치 + 의존성 (모델 실행 extra 포함). 한글 그래프용 글꼴도 설치
!pip install -q uv
!apt-get install -qq -y fonts-nanum > /dev/null
!uv sync --extra models
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"


In [ ]:
# 4. 데이터 연결. 드라이브 폴더를 AI/data 아래에 심볼릭 링크 (복사 없음)
!mkdir -p data
for name in ("musdb18hq", "gmd", "synth_v1"):
    src = f"{DRIVE}/data/{name}"
    dst = f"data/{name}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
!uv run ssaplash-bench check-data configs/bench/datasets/musdb18hq_test.yaml configs/bench/datasets/synth_v1.yaml


In [ ]:
# 5. 동작 확인: 모델 1개 × 30초 클립 3곡
!PYTHONUTF8=1 uv run ssaplash-bench run     --models configs/bench/models/scnet_large.yaml     --dataset configs/bench/datasets/musdb18hq_test_clip30.yaml     --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns"


In [ ]:
# 6. 본측정: 모델 전부 × MUSDB 테스트 50곡 (수 시간). 결과 CSV는 곡마다 바로 저장되므로 끊겨도 남는다
MODELS = " ".join(f"configs/bench/models/{m}.yaml" for m in [
    "scnet_xl_ihf", "bs_roformer_sw", "htdemucs_ft",       # 1순위
    "scnet_large", "bs_roformer_musdb",                     # 2순위
    "htdemucs", "umxhq",                                    # 하한선 (spleeter는 별도 환경)
])
!PYTHONUTF8=1 uv run ssaplash-bench run --models $MODELS     --dataset configs/bench/datasets/musdb18hq_test.yaml     --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" --save-wav 3


In [ ]:
# 7. 본측정: 합성곡 5곡 (정답 온셋이 MIDI)
!PYTHONUTF8=1 uv run ssaplash-bench run --models $MODELS     --dataset configs/bench/datasets/synth_v1.yaml     --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" --save-wav 5


In [ ]:
# 8. 두 실행을 합친 보고서. <runId>는 6·7의 출력에서 복사
!ls "$DRIVE/results"
# !PYTHONUTF8=1 uv run ssaplash-bench report "$DRIVE/results/<musdb runId>/results.csv" "$DRIVE/results/<synth runId>/results.csv" --out "$DRIVE/results/combined"


로컬에서 보고서를 다시 만들려면 드라이브의 `results/<runId>/results.csv`를 받아 `uv run ssaplash-bench report <csv>`를 돌린다.
MLflow 기록은 `mlruns/` 폴더째 받아 `uv run mlflow ui --backend-store-uri mlruns`로 본다.
